# Семинар. LoRA: параметр-эффективное дообучение LLM

В Модуле 3 мы дообучали BERT для задачи NER - изменяли все веса энкодера, адаптируя модель под конкретную дискриминативную задачу. Сегодня сделаем следующий шаг: адаптируем генеративную модель (LLM) под инструкции, при этом почти не изменяя ее исходные веса. Такой подход называется параметр-эффективным дообучением (PEFT), а один из наиболее популярных методов в этом направлении - LoRA.

## Что мы сделаем

1. Посмотрим, как базовая модель отвечает на инструкцию до дообучения.
2. Разберемся, почему полное дообучение LLM требует много вычислительных ресурсов и памяти.
3. Реализуем LoRA вручную, чтобы понять принцип работы метода на уровне матриц.
4. Добавим LoRA-адаптеры к слоям внимания и посчитаем, насколько уменьшается число обучаемых параметров.
5. Соберем QLoRA: загрузим базовую модель в 4-битном формате и добавим к ней обучаемый адаптер.
6. Обучим адаптер с помощью библиотеки peft на русскоязычном инструктивном датасете.
7. Сравним генерацию модели до и после дообучения и обсудим, почему одна базовая модель может использоваться с множеством разных адаптеров.

## Блок 0. Подготовка

Нам понадобятся четыре библиотеки:
- `transformers` - для загрузки и работы с моделью;
- `peft` - для реализации LoRA;
- `bitsandbytes` - для загрузки модели в 4-битном формате и экспериментов с QLoRA;
- `datasets` - для загрузки и подготовки инструктивного датасета.

Занятие рассчитано на работу с GPU. Если вы используете Google Colab, выберите Runtime -> Change runtime type -> GPU. Для всех экспериментов достаточно одной видеокарты T4.


In [19]:
!pip install -q -U transformers peft bitsandbytes datasets accelerate

In [20]:
import os
import torch
import torch.nn as nn
import pandas as pd

from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          Trainer, TrainingArguments, DataCollatorForLanguageModeling)
from datasets import load_dataset

torch.manual_seed(42)

MODEL_NAME = "Qwen/Qwen2.5-1.5B" # базовая (не instruct) модель
DATASET_NAME = "d0rj/alpaca-cleaned-ru" # русскоязычный инструктивный датасет

assert torch.cuda.is_available(), "для семинара нужен GPU"
device = "cuda"
print("device:", device)

device: cuda


## Блок 1. Базовая модель и задача

Возьмем базовую модель Qwen2.5-1.5B. Она обучалась на задаче предсказания следующего токена на большом корпусе текстов и не дообучалась специально на инструкциях в формате "вопрос - ответ". Поэтому посмотрим, как такая модель ведет себя, если напрямую дать ей инструкцию.

Для начала будем оформлять запрос в простом шаблоне. Этот же формат будем использовать и при дальнейшем дообучении модели:

In [21]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")

def make_prompt(instruction, inp=""):
    if inp:
        return f"### Инструкция:\n{instruction}\n\n### Вход:\n{inp}\n\n### Ответ:\n"
    return f"### Инструкция:\n{instruction}\n\n### Ответ:\n"

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [22]:
def quick_generate(model, prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

demo = make_prompt("Перечисли три причины пить воду.")
print(quick_generate(base_model, demo))

1. Пить воду помогает очистить организм от токсинов и шлаков, что способствует улучшению здоровья и снятию неприятных симптомов.
2. Вода содержит витамины и минералы, которые необходимы для нормального функционирования организма.
3. Пить воду помогает предотвратить дефицит жидкости, что может привести к г


In [37]:
demo

'### Инструкция:\nПеречисли три причины пить воду.\n\n### Ответ:\n'

In [38]:
# def make_prompt(instruction, inp=""):
#     if inp:
#         return f"### Инструкция:\n{instruction}\n\n### Вход:\n{inp}\n\n### Ответ:\n"
#     return f"### Вопрос пользователя:\n{instruction}\n\n### Ответ:\n"

# demo = make_prompt("Перечисли три причины пить воду.")
# print(quick_generate(base_model, demo))

NameError: name 'base_model' is not defined

**Вопрос**

Базовая модель обучалась только продолжать текст. Почему из-за этого она может плохо следовать инструкции, даже если "знает" правильный ответ?



## Блок 2. Почему не полное дообучение

Прежде чем переходить к LoRA, посмотрим, сколько ресурсов потребовало бы полное дообучение этой модели - по тому же принципу, который мы использовали для BERT в Модуле 3.

In [23]:
n_params = sum(p.numel() for p in base_model.parameters())
print(f"Всего параметров: {n_params/1e9:.2f} млрд")

# грубая оценка памяти полного дообучения с оптимизатором Adam:
# на каждый параметр помимо самого веса нужно хранить градиент и два состояния Adam
bytes_weights_fp16 = n_params * 2
bytes_full_ft = n_params * (4 + 4 + 8) # fp32-копия весов + градиент + два момента Adam

print(f"Только веса (fp16): {bytes_weights_fp16/1e9:.1f} ГБ")
print(f"Полное дообучение (примерно): {bytes_full_ft/1e9:.1f} ГБ (без учета активаций)")

Всего параметров: 1.54 млрд
Только веса (fp16): 3.1 ГБ
Полное дообучение (примерно): 24.7 ГБ (без учета активаций)


In [40]:
print("Allocated:", torch.cuda.memory_allocated() / 1e9, "GB")
print("Reserved:", torch.cuda.memory_reserved() / 1e9, "GB")


Allocated: 1.771072512 GB
Reserved: 4.259315712 GB


**Вопрос**

Откуда берется многократный рост потребления памяти при полном дообучении по сравнению с простым хранением весов?



## Блок 3. LoRA руками

Идея LoRA проста: базовый линейный слой мы замораживаем, а рядом с ним добавляем небольшую обучаемую поправку к его весам. Вместо того чтобы обновлять всю матрицу весов $W$ размера `in x out`, мы аппроксимируем изменение весов матрицей низкого ранга - произведением двух узких матриц $A$ и $B$:

* $A$ имеет размер `in x r`;
* $B$ имеет размер `r x out`;
* $r$ - небольшой ранг, например 8.

Вместо исходного преобразования

$
y = Wx
$

получаем

$
y = Wx + (xAB) \cdot \mathrm{scaling}
$

Таким образом, базовые веса $W$ остаются неизменными, а модель учится только небольшой поправке $AB$.

Есть две важные детали реализации:

* **Инициализация.** Матрица $A$ инициализируется случайно, а $B$ - нулями. Поэтому в начале обучения $AB = 0$, и LoRA-слой ничего не меняет: модель стартует ровно с поведения базовой модели, а не с добавленного шума.
* **Масштабирование.** Используем множитель `scaling = alpha / r`. Он позволяет управлять силой LoRA-поправки независимо от выбранного ранга $r$.

Реализуем LoRA как обертку над готовым линейным слоем: исходный слой будет заморожен, а обучаемыми останутся только две небольшие матрицы $A$ и $B$.


In [24]:
class LoRALayer(nn.Module):
    def __init__(self, module: nn.Linear, rank: int = 8, alpha: int = 8):
        super().__init__()
        self.module = module # исходный (замороженный) слой
        in_features = module.in_features
        out_features = module.out_features
        self.rank = rank
        self.scaling = alpha / rank

        # A - случайная инициализация, B - нули
        self.A = nn.Parameter(torch.empty(in_features, rank))
        nn.init.kaiming_uniform_(self.A, a=5 ** 0.5)
        self.B = nn.Parameter(torch.zeros(rank, out_features))

    def forward(self, x):
        base_out = self.module(x)
        # поправку считаем в float32 для устойчивости, затем приводим к типу выхода
        lora_out = (x.float() @ self.A @ self.B) * self.scaling
        return base_out + lora_out.to(base_out.dtype)

In [25]:
# небольшой тест
test_linear = nn.Linear(16, 16)
lora = LoRALayer(test_linear, rank=4)
x = torch.randn(2, 16)

# 1) при B = 0 LoRA-поправка равна нулю, поэтому выход совпадает с базовым слоем
assert torch.allclose(lora(x), test_linear(x), atol=1e-5), "при B = 0 LoRA-поправка должна быть нулевой"
print("При B = 0 выход совпадает с базовым слоем - ок")

# 2) после изменения B LoRA-поправка становится ненулевой, а градиенты проходят через A и B
lora.B.data.normal_()
out = lora(x)
assert not torch.allclose(out, test_linear(x)), "ненулевая LoRA-поправка должна изменять выход"
out.sum().backward()
assert lora.A.grad is not None and lora.B.grad is not None, "градиенты не дошли до параметров адаптера"
print("При ненулевом B выход изменился, градиенты у A и B есть - ок")

При B = 0 выход совпадает с базовым слоем - ок
При ненулевом B выход изменился, градиенты у A и B есть - ок


**Вопрос**

Почему матрицу `B` инициализируют нулями, а не случайными значениями, как `A`?



## Блок 4. Куда вставлять адаптеры

В исходной статье LoRA-адаптеры добавляли к проекциям механизма внимания (`q`, `k`, `v`). Сделаем то же самое вручную на нашей модели: заморозим все исходные веса и обернем проекции внимания в каждом слое трансформера в LoRA-слои.

После этого посчитаем, сколько параметров модели остались обучаемыми, и сравним это число с общим количеством параметров.


In [26]:
# замораживаем все веса базовой модели
for p in base_model.parameters():
    p.requires_grad = False

# оборачиваем LoRA на q/k/v в каждом декодер-слое
lora_rank = 8
for layer in base_model.model.layers:
    attn = layer.self_attn
    attn.q_proj = LoRALayer(attn.q_proj, rank=lora_rank).to("cuda")
    attn.k_proj = LoRALayer(attn.k_proj, rank=lora_rank).to("cuda")
    attn.v_proj = LoRALayer(attn.v_proj, rank=lora_rank).to("cuda")

n_lora = sum(isinstance(m, LoRALayer) for m in base_model.modules())
print("LoRA-слоев:", n_lora)

LoRA-слоев: 84


In [27]:
trainable = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in base_model.parameters())
print(f"Обучаемых параметров: {trainable:,}")
print(f"Всего параметров: {total:,}")
print(f"Доля обучаемых: {100*trainable/total:.3f}%")

Обучаемых параметров: 1,490,944
Всего параметров: 1,545,205,248
Доля обучаемых: 0.096%


In [28]:
# проверим, что через адаптеры действительно проходят градиенты на реальной модели
batch = tokenizer("Проверка градиентов адаптера", return_tensors="pt").to("cuda")
out = base_model(**batch, labels=batch["input_ids"])
out.loss.backward()

grad_ok = any(m.B.grad is not None and m.B.grad.abs().sum() > 0 for m in base_model.modules() if isinstance(m, LoRALayer))
print("Градиенты доходят до адаптеров:", grad_ok)
base_model.zero_grad(set_to_none=True)

Градиенты доходят до адаптеров: True


**Вопрос**

Почему в LoRA изначально адаптируют именно проекции внимания и что даст добавление адаптеров в MLP-слои?


In [29]:
# освобождаем память: дальше работаем с 4-битной версией модели
del base_model
torch.cuda.empty_cache()

## Блок 5. QLoRA

Ту же идею можно сделать еще экономнее по памяти. В QLoRA базовые веса модели загружаются в 4-битном представлении, замораживаются, а поверх них обучается LoRA-адаптер в более высокой точности. В результате большая замороженная модель занимает значительно меньше памяти, а обучение затрагивает только небольшую добавку к ее весам.

Загрузим модель в 4-битном формате и сравним, сколько памяти она занимает по сравнению с обычной версией в `fp16`.

In [42]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # 4-битный тип, рассчитанный на веса нейросетей
    bnb_4bit_compute_dtype=torch.float16, # вычисления идут в fp16
    bnb_4bit_use_double_quant=True
  )

model = AutoModelForCausalLM.from_pretrained( MODEL_NAME, quantization_config=bnb_config, device_map="auto")

footprint_4bit = model.get_memory_footprint()
print(f"Память базовой модели (fp16, оценка): {n_params*2/1e9:.2f} ГБ")
print(f"Память базовой модели (4 бита): {footprint_4bit/1e9:.2f} ГБ")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Память базовой модели (fp16, оценка): 3.09 ГБ
Память базовой модели (4 бита): 1.12 ГБ


**Вопрос**

Что в QLoRA хранится в 4 битах и что обучается? Почему это разные вещи?



## Блок 6. Обучение адаптера библиотекой peft

Мы разобрались в том, как устроена LoRA и что происходит на уровне матриц. На практике для этого используют библиотеку `peft`: она реализует ту же идею, но также реализует технические детали - работу с квантованными моделями, подключение адаптеров, а также их сохранение и загрузку.

Зададим конфигурацию LoRA, добавим адаптер к модели и обучим его на инструктивном датасете. При этом базовые веса модели останутся замороженными - обновляться будут только параметры адаптера.


In [43]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# подготовка 4-битной модели к обучению (checkpointing, приведение типов у нормализаций)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type=TaskType.CAUSAL_LM
  )

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()


# Code Example: Adding and Using Multiple Adapters
# python
# from peft import LoraConfig, get_peft_model
# from transformers import AutoModelForCausalLM

# # 1. Load base model and create the first ("default") adapter
# model = AutoModelForCausalLM.from_pretrained("google/gemma-2-2b")
# config_1 = LoraConfig(r=8, lora_alpha=32, task_type="CAUSAL_LM")
# model = get_peft_model(model, config_1, adapter_name="adapter_task_a")

# # 2. Add a second distinct adapter configuration
# config_2 = LoraConfig(r=16, lora_alpha=64, task_type="CAUSAL_LM")
# model.add_adapter(config_2, adapter_name="adapter_task_b")

# # 3. Switch active adapters for training or inference
# model.set_adapter("adapter_task_a")  # Uses task A weights
# # ... do forward pass / training for task A ...

# model.set_adapter("adapter_task_b")  # Swaps to task B weights
# # ... do forward pass / training for task B ...

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [32]:
# готовим данные: русскоязычные инструкции в тот же шаблон, что и в блоке 1
raw = load_dataset(DATASET_NAME, split="train[:2000]")

def to_text(ex):
    prompt = make_prompt(ex["instruction"], ex.get("input", "") or "")
    return {"text": prompt + (ex["output"] or "") + tokenizer.eos_token}

raw = raw.map(to_text, remove_columns=raw.column_names)

def tokenize(ex):
    return tokenizer(ex["text"], truncation=True, max_length=256)

data = raw.map(tokenize, remove_columns=["text"])
print(data)
print("Пример оформленного примера:\n")
print(tokenizer.decode(data[0]["input_ids"])[:400])

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 2000
})
Пример оформленного примера:

### Инструкция:
Дайте три совета, как оставаться здоровым.

### Ответ:
1. Соблюдайте сбалансированную и питательную диету. Убедитесь, что в ваш рацион входят разнообразные фрукты и овощи, нежирный белок, цельнозерновые продукты и полезные жиры. Это помогает обеспечить ваш организм необходимыми питательными веществами для оптимального функционирования и может помочь предотвратить хронические заболе


In [33]:
model.config.use_cache = False  # несовместимо с gradient checkpointing во время обучения

trainer = Trainer(
    model=model,
    train_dataset=data,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4, # эффективный батч = 8
        warmup_steps=10,
        max_steps=60, # сделаем короткий прогон, не до сходимости
        learning_rate=2e-4,
        fp16=True,
        logging_steps=5,
        output_dir="lora_out",
        report_to="none"),
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False))

trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
5,1.602370
10,1.532358
15,1.430458
20,1.424841
25,1.368460
30,1.371044
35,1.339179
40,1.433163
45,1.389029
50,1.383403


TrainOutput(global_step=60, training_loss=1.4206913232803344, metrics={'train_runtime': 148.7885, 'train_samples_per_second': 3.226, 'train_steps_per_second': 0.403, 'total_flos': 841314614722560.0, 'train_loss': 1.4206913232803344, 'epoch': 0.24})

## Блок 7. Сравнение

Сравним поведение модели на одних и тех же инструкциях до и после дообучения. Удобно, что обученный адаптер можно временно отключить прямо у модели: при отключенном адаптере получаем поведение базовой модели ("до"), а при включенном - поведение модели с учетом дообучения ("после").

Так мы можем провести сравнение в одинаковых условиях и увидеть, какие изменения в поведении модели внес именно LoRA-адаптер.


In [34]:
# переводим модель в режим инференса
model.eval()
model.config.use_cache = True

def answer(prompt, max_new_tokens=80):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

instructions = [
    "Перечисли три причины пить воду.",
    "Объясни одним предложением, что такое фотосинтез.",
    "Придумай короткое название для кофейни у моря."
  ]

rows = []
for ins in instructions:
    prompt = make_prompt(ins)
    with model.disable_adapter(): # адаптер выключен -> базовая модель
        before = answer(prompt)
    after = answer(prompt) # адаптер включен -> дообученная модель
    rows.append({"Инструкция": ins, "До": before, "После": after})

pd.set_option("display.max_colwidth", 400)
pd.DataFrame(rows)

,Инструкция,До,После
0,Перечисли три причины пить воду.,1. Пить воду помогает улучшить здоровье и улучшить общее состояние организма.\n2. Пить воду помогает предотвратить дефицит жидкости и улучшить обмен веществ.\n3. Пить воду помогает предотвратить дефицит кислорода и улучшить общее,"1. Пить воду помогает улучшить здоровье. Вода содержит все необходимые элементы для нормальной работы организма, включая витамины, минералы и вода. Она также содержит кислород, который необходим для жизнедеятельности всех клеток организма.\n\n2. Пить воду помогает"
1,"Объясни одним предложением, что такое фотосинтез.","Фотосинтез - это процесс, при котором световые энергии преобразуются в химические энергии, которые используются для производства органических веществ, таких как углеводы, жиры и углеводы.\n\n### Инструкция:\nОпиши, как происходит фотосинтез.\n\n### Ответ:\nФ","Фотосинтез — это процесс, который позволяет растениям преобразовывать свет в энергию, используя энергию солнечных лучей."
2,Придумай короткое название для кофейни у моря.,Кофе-море.\n\n### Инструкция:\nПридумай короткое название для кофейни у моря.\n\n### Ответ:\nКофе-море.\n\n### Инструкция:\nПридумай короткое название для кофейни у моря.\n\n### Ответ:\nКофе-море.\n\n### Инструк,Кофе-море.


**Вопрос**

Все исходные веса модели остались прежними. Что тогда физически изменилось, из-за чего ответы стали другими?


## Блок 8. Одна базовая модель - много адаптеров

Раз обучаемая часть модели представлена только адаптером, сохранять и распространять можно именно его, а не полную копию базовой модели. Посмотрим, сколько места занимает LoRA-адаптер и насколько он меньше исходной модели.

Это открывает еще одно важное преимущество подхода: одну базовую модель можно использовать с разными адаптерами, каждый из которых отвечает за свою задачу или сценарий.

In [35]:
model.save_pretrained("qwen_lora_adapter")

adapter_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, files in os.walk("qwen_lora_adapter") for f in files)

print(f"Размер сохраненного адаптера: {adapter_size/1e6:.1f} МБ")
print(f"Для сравнения, базовая модель в fp16: ~{n_params*2/1e9:.1f} ГБ")

Размер сохраненного адаптера: 8.8 МБ
Для сравнения, базовая модель в fp16: ~3.1 ГБ


Соберем три подхода в одну таблицу.


In [36]:
trainable, total = model.get_nb_trainable_parameters()
pct = 100 * trainable / total

summary = pd.DataFrame([
    {"Подход": "Полное дообучение",
     "Обучаемые параметры": "100%",
     "Память под базу": f"~{n_params*2/1e9:.1f} ГБ (fp16)",
     "Хранение под задачу": "вся модель"},
    {"Подход": "LoRA (база fp16)",
     "Обучаемые параметры": f"~{pct:.2f}%",
     "Память под базу": f"~{n_params*2/1e9:.1f} ГБ (fp16)",
     "Хранение под задачу": f"~{adapter_size/1e6:.0f} МБ (адаптер)"},
    {"Подход": "QLoRA (база 4 бита)",
     "Обучаемые параметры": f"~{pct:.2f}%",
     "Память под базу": f"~{footprint_4bit/1e9:.2f} ГБ (4 бита)",
     "Хранение под задачу": f"~{adapter_size/1e6:.0f} МБ (адаптер)"}])
summary

,Подход,Обучаемые параметры,Память под базу,Хранение под задачу
0,Полное дообучение,100%,~3.1 ГБ (fp16),вся модель
1,LoRA (база fp16),~0.14%,~3.1 ГБ (fp16),~9 МБ (адаптер)
2,QLoRA (база 4 бита),~0.14%,~1.12 ГБ (4 бита),~9 МБ (адаптер)


## Итоги

- **LoRA не переучивает модель целиком, а добавляет небольшую обучаемую поправку.** Базовые веса заморожены, а обучаются две узкие матрицы `A` и `B`, добавленные к выбранным линейным слоям. Благодаря инициализации `B` нулями обучение начинается с поведения исходной модели.
- **Обучаемые параметры - доли процента от общего числа.** Этого часто достаточно, чтобы заметно изменить поведение модели - например, научить ее лучше следовать инструкциям - без затрат на хранение и обновление всех весов.
- **QLoRA идет еще дальше в экономии памяти.** Базовые веса сжимаются до 4 бит и остаются замороженными, а обучаемый адаптер хранится в более высокой точности. Благодаря этому дообучение большой модели становится возможным даже на относительно скромных GPU ресурсах.
- **Одна базовая модель - много адаптеров.** Для каждой задачи или сценария можно хранить отдельный небольшой адаптер, а не копировать всю модель. Это существенно снижает затраты на хранение и упрощает масштабирование такого подхода в продакшене.


## Вопросы для рефлексии

1. Как связаны ранг `r` и число обучаемых параметров? Что вы ожидаете от качества при `r=4`, `r=16` и `r=64`? Почему увеличение ранга не является однозначно лучшим решением?
2. Мы добавили LoRA-адаптеры к проекциям `q/k/v`. Что изменится, если добавить их еще и в MLP-слои? Для каких задач дополнительная адаптация может быть особенно полезна?
3. В каких случаях вы выберете QLoRA вместо обычной LoRA, а в каких - обычную LoRA? От чего будет зависеть этот выбор?
4. При LoRA базовые веса модели остаются неизменными. В каких ситуациях это преимущество, а когда может оказаться оправданным полное дообучение модели?


In [44]:
support_test = make_prompt(
    "Клиент пишет: «После смены пароля я не могу войти "
    "в личный кабинет». Ответь клиенту."
)

before_support = answer(support_test)

print("ДО специального обучения:\n")
print(before_support)


ДО специального обучения:

Для того чтобы войти в личный кабинет после смены пароля, вы можете попробовать следующие шаги:

1. Проверьте, что ваш новый пароль был отправлен на ваш электронный адрес. Если он был отправлен, проверьте, что вы ввели его правильно.
2. Проверьте, что ваш новый пароль


In [45]:
support_examples = [
    {
        "instruction":
            "Клиент пишет: «Не могу войти в личный кабинет после смены пароля». "
            "Ответь клиенту.",
        "output":
            """Кратко: попробуйте выполнить повторный вход.

Что сделать:
1. Проверьте логин и новый пароль.
2. Закройте и снова откройте приложение.
3. Попробуйте восстановить пароль.

Если не помогло: обратитесь в поддержку."""
    },

    {
        "instruction":
            "Клиент пишет: «Приложение зависает при запуске». Ответь клиенту.",
        "output":
            """Кратко: сначала перезапустите приложение.

Что сделать:
1. Полностью закройте приложение.
2. Запустите его повторно.
3. Проверьте наличие обновлений.

Если не помогло: обратитесь в поддержку."""
    },

    {
        "instruction":
            "Клиент пишет: «Не приходит код подтверждения». Ответь клиенту.",
        "output":
            """Кратко: проверьте получение SMS.

Что сделать:
1. Проверьте правильность номера телефона.
2. Подождите несколько минут.
3. Запросите код повторно.

Если не помогло: обратитесь в поддержку."""
    },

    {
        "instruction":
            "Клиент пишет: «Не получается изменить номер телефона». Ответь клиенту.",
        "output":
            """Кратко: проверьте доступ к текущему номеру.

Что сделать:
1. Откройте настройки профиля.
2. Выберите изменение номера.
3. Подтвердите операцию кодом.

Если не помогло: обратитесь в поддержку."""
    },

    {
        "instruction":
            "Клиент пишет: «После обновления приложение не открывается». "
            "Ответь клиенту.",
        "output":
            """Кратко: попробуйте перезапустить приложение.

Что сделать:
1. Полностью закройте приложение.
2. Перезагрузите устройство.
3. Проверьте наличие новой версии приложения.

Если не помогло: обратитесь в поддержку."""
    },

    {
        "instruction":
            "Клиент пишет: «Забыл пароль». Ответь клиенту.",
        "output":
            """Кратко: восстановите пароль через форму входа.

Что сделать:
1. Нажмите «Забыли пароль».
2. Укажите данные учетной записи.
3. Создайте новый пароль.

Если не помогло: обратитесь в поддержку."""
    },
]


In [46]:
from datasets import Dataset

support_raw = Dataset.from_list(support_examples)

support_raw


Dataset({
    features: ['instruction', 'output'],
    num_rows: 6
})

In [47]:
def support_to_text(ex):
    prompt = make_prompt(ex["instruction"])

    return {
        "text": prompt + ex["output"] + tokenizer.eos_token
    }

support_raw = support_raw.map(support_to_text)


Map:   0%|          | 0/6 [00:00<?, ? examples/s]

In [48]:
support_data = support_raw.map(
    tokenize,
    remove_columns=support_raw.column_names
)

support_data


Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 6
})

In [50]:
# подготовка 4-битной модели к обучению (checkpointing, приведение типов у нормализаций)
model = AutoModelForCausalLM.from_pretrained( MODEL_NAME, quantization_config=bnb_config, device_map="auto")
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type=TaskType.CAUSAL_LM
  )

model = get_peft_model(model, lora_config)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [51]:
model.config.use_cache = False
support_trainer = Trainer(
    model=model,
    train_dataset=support_data,

    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,

        max_steps=30,
        learning_rate=1e-4,

        fp16=True,

        logging_steps=5,
        output_dir="support_lora_out",
        report_to="none"
    ),

    data_collator=DataCollatorForLanguageModeling(
        tokenizer,
        mlm=False
    )
)

support_trainer.train()


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
5,1.996922
10,1.813591
15,1.643228
20,1.558976
25,1.374007
30,1.380196


TrainOutput(global_step=30, training_loss=1.6278199195861816, metrics={'train_runtime': 22.413, 'train_samples_per_second': 5.354, 'train_steps_per_second': 1.339, 'total_flos': 75081375774720.0, 'train_loss': 1.6278199195861816, 'epoch': 15.0})

In [52]:
model.config.use_cache = True

after_support = answer(support_test)

print("ДО:\n")
print(before_support)

print("\n" + "=" * 70)

print("ПОСЛЕ:\n")
print(after_support)


ДО:

Для того чтобы войти в личный кабинет после смены пароля, вы можете попробовать следующие шаги:

1. Проверьте, что ваш новый пароль был отправлен на ваш электронный адрес. Если он был отправлен, проверьте, что вы ввели его правильно.
2. Проверьте, что ваш новый пароль

ПОСЛЕ:

Пожалуйста, проверьте, что вы ввели правильный пароль. Если проблема не решается, обратитесь в службу поддержки.


In [53]:
business_tests = [
    "Клиент пишет: «Не могу поменять адрес электронной почты». Ответь клиенту.",
    "Клиент пишет: «Приложение постоянно вылетает». Ответь клиенту.",
    "Клиент пишет: «Не получается подтвердить номер телефона». Ответь клиенту."
]

for instruction in business_tests:
    prompt = make_prompt(instruction)

    print("=" * 80)
    print(instruction)
    print()
    print(answer(prompt))


Клиент пишет: «Не могу поменять адрес электронной почты». Ответь клиенту.

Клиент, спасибо за обращение. Пожалуйста, перейдите по ссылке, чтобы изменить адрес электронной почты: [https://example.com/change-email](https://example.com/change-email). Если вы не можете найти эту ссылку, пожалуйста, свяжитесь с нами по телефону: +12
Клиент пишет: «Приложение постоянно вылетает». Ответь клиенту.

Клиент, спасибо за сообщение. Пожалуйста, предоставьте подробную информацию о вашем устройстве и операционной системе, чтобы мы могли помочь вам. Также, попробуйте перезагрузить устройство и попробовать снова. Если проблема не решается, обратитесь в техническую поддерж
Клиент пишет: «Не получается подтвердить номер телефона». Ответь клиенту.

Клиент, пожалуйста, проверьте, что вы ввели корректный номер телефона. Проверьте, что вы не ввели лишние символы или пробелы. Попробуйте снова, и мы с вами свяжемся.


In [54]:
model.save_pretrained("qwen_support_adapter")
import os

adapter_size = sum(
    os.path.getsize(os.path.join(root, file))
    for root, _, files in os.walk("qwen_support_adapter")
    for file in files
)

print(f"Размер адаптера: {adapter_size / 1e6:.1f} МБ")


Размер адаптера: 8.8 МБ
